<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor experiment

## Why this problem?

A committor is the probability that a stochastic trajectory reaches a target set before it returns to a source set. It is a central quantity in transition path theory and metastability: it describes which way a transition is likely to go from each starting point. This example tests a boundary-adapted neural representation on a small periodic problem with two separate interior boundaries.

This is a **committor** experiment, not a mean escape time experiment. The committor answers a probability question; a mean escape time answers how long a process typically takes to hit a set. The mathematical motivation for boundary-adapted PINNs is discussed in the accompanying paper, [*Boundary-Adapted PINNs for Elliptic Dirichlet Problems*](https://arxiv.org/abs/2607.19167).

## Stochastic dynamics and the committor equation

Let $X_t$ follow overdamped Langevin dynamics on the flat torus $\mathbb{T}^2=[0,1)^2$:
$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t\pmod 1,
\qquad
V(x_1,x_2)=a[1-\cos(4\pi(x_1-1/4))]+b[1-\cos(2\pi x_2)].
$$
The periodic disks $A$ and $B$ have centers $c_A=(1/4,0)$ and $c_B=(3/4,0)$ and common radius $r$. For their first hitting times $\tau_A$ and $\tau_B$, the forward committor is
$$
q(x)=\mathbb{P}_x(\tau_B<\tau_A).
$$
It is the probability of reaching $B$ before $A$ when starting at $x$. The diffusion generator is
$$
\mathcal{L}f=-\nabla V\cdot\nabla f+\varepsilon\Delta f.
$$
The Markov property implies that $q$ is harmonic with respect to this generator:
$$
\mathcal{L}q=0\quad\text{in }D=\mathbb{T}^2\setminus(A\cup B),
\qquad q|_{\partial A}=0,\quad q|_{\partial B}=1.
$$
The default parameters are $a=1$, $b=0.6$, and $\varepsilon=0.4$.

## Exact boundary values from a smooth geometry factor

For a periodic disk centered at $c_j$, the code uses the smooth periodic defining function
$$
\omega_j(x)=4\sum_{\ell=1}^2\sin^2(\pi(x_\ell-c_{j\ell}))-r^2.
$$
It is negative inside the disk, zero on its boundary, and positive in the exterior domain. The normalized defining function is
$$
d_j(x)=\frac{\omega_j(x)}{\sqrt{\omega_j(x)^2+|\nabla\omega_j(x)|^2}}.
$$
On the boundary, $d_j=0$ and $|\nabla d_j|=1$. In $D$, both $d_A$ and $d_B$ are positive. Define the boundary lift $g$ and the two-boundary factor $\rho$ by
$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},\qquad
\rho(x)=\frac{d_A(x)d_B(x)}{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$
The network represents
$$
q_\theta(x)=g(x)+\rho(x)N_\theta(x).
$$
On $\partial A$, $g=0$ and $\rho=0$; on $\partial B$, $g=1$ and $\rho=0$. Therefore the two Dirichlet values hold for every choice of network weights. Near either boundary, $\rho$ has the first-order scaling of that boundary's normalized defining function. The simpler product $d_Ad_B$ also vanishes on both boundaries, but its normal scaling depends on the distance to the other boundary.

All three networks receive sine and cosine coordinates $(\sin(2\pi x_1),\sin(2\pi x_2),\cos(2\pi x_1),\cos(2\pi x_2))$, so their outputs are periodic by construction.

## Methods and numerical comparison

The experiment compares three representations:

1. **Boundary-adapted:** exact lift $g$ and normalized factor $\rho$.
2. **Naïve product:** the same lift with the unnormalized product $d_Ad_B$ in place of $\rho$.
3. **Penalty:** a periodic multilayer perceptron trained with sampled boundary penalty terms.

A finite-difference solution on periodic grids provides a numerical reference. Comparing two grid resolutions helps estimate reference discretization error; it is a numerical check, not exact ground truth. The saved figures show the three learned committors, the reference, and pointwise errors. The summary reports grid-based errors and boundary diagnostics.

## Further reading

- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- Y. Gao, T. Li, X. Li, and J.-G. Liu, “Transition path theory for Langevin dynamics on manifold: optimal control and data-driven solver,” [arXiv:2010.09988](https://arxiv.org/abs/2010.09988) (2020).

## What the notebook cells do

| Stage | Purpose | Hardware | What it tells you |
|---|---|---|---|
| Setup and CPU smoke tests | Fetches current `main`, installs the package, and runs tests, including a tiny one-epoch comparison through all three methods. | CPU | Checks reference generation, training, checkpoints, and summaries. No GPU is used. |
| CPU preflight | Runs all three methods for two epochs with a tiny network and 32/64-grid references. | CPU | Checks that training, reference solve, checkpoints, summaries, and figures work together. The output is a pipeline check, not a scientific result. |
| GPU pilot | Runs all three methods for 6,000 epochs each, with 128/256-grid references. | GPU | A substantive, single-seed exploratory comparison. It uses GPU time; it is not a multi-seed or final paper result. It starts only when you run its cell. |
| Inspect or resume | Displays saved results or resumes interrupted methods from Drive checkpoints. | No training to inspect; GPU to resume | Inspection is cheap. Resuming continues the existing GPU run. |

### GitHub token

While the repository is private, the setup cell uses your saved `GITHUB_TOKEN` Colab Secret to fetch the latest code from the repository's `main` branch. It refreshes only the temporary clone under `/content/boundary-adapted-pinns`. The token grants repository read access; the numerical code does not use it, and it is not saved to Drive or to the repository. You already created it for this private-repository fetch, so keep it in Colab Secrets while the repository is private. It uses no GPU time. If the repository becomes public, setup can fetch the code without the token.

## 1. Setup and CPU tests — CPU only

Run this cell first. It fetches the current repository into Colab's `/content` directory, installs the Python package, and runs the tests. One test runs a tiny one-epoch CPU comparison through all three methods and checks that it creates the reference, checkpoints, and summary. This is a fast pipeline check, not a scientific experiment; it uses no GPU. The printed code revision is recorded with your run outputs.

In [ ]:
# Setup only: fetch current main, install the package, and run tests. This cell does not train a PINN.
from pathlib import Path
import base64
import os
import subprocess

from google.colab import drive, userdata
drive.mount('/content/drive')

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Stream command output to Colab and save it in the Drive run folder."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        raise RuntimeError(
            f"Command failed with exit status {return_code}. Full output is saved at {log_path}"
        )

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Public repositories can be cloned without a Colab Secret.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch main from GitHub. If this repository is private, add the GITHUB_TOKEN '
        'Colab Secret and enable Notebook access. The token is only used to read the repository.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Setup and CPU tests passed. Code revision:', revision)


## 2. CPU preflight — complete pipeline, negligible scientific cost

Run this before any GPU work. It executes the same comparison command with a tiny network, two epochs, and 32/64-grid references. It saves the command output to `console.log` in its Drive folder. The GPU pilot and resume cells are locked unless this cell finishes successfully in the current runtime.

In [ ]:
# CPU only: full pipeline check using the tiny preflight configuration.
from datetime import datetime
from pathlib import Path
import subprocess

CPU_PREFLIGHT_PASSED = False
preflight_name = 'preflight_' + datetime.now().strftime('%Y%m%d_%H%M%S')
PREFLIGHT_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{preflight_name}'
Path(PREFLIGHT_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preflight results:', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], Path(PREFLIGHT_ROOT) / 'console.log')
if not Path(PREFLIGHT_ROOT, 'summary.json').is_file():
    raise RuntimeError(f'CPU preflight did not produce summary.json. Inspect {PREFLIGHT_ROOT}/console.log')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. Inspect the saved summary and figures before considering the GPU pilot.')

## 3. GPU pilot — first substantive numerical experiment

The GPU cell runs only after the CPU preflight succeeds in this runtime. It uses 64-wide, 4-hidden-layer networks; 1,024 interior points; 6,000 epochs per method; and 128/256-grid references. It saves checkpoints and its console output to Drive.

In [ ]:
# GPU required: full three-method committor pilot.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('GPU pilot blocked: the CPU preflight has not passed in this runtime. Fix its error first.')
from datetime import datetime
from pathlib import Path
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('The GPU pilot requires a T4 GPU. Select it under Runtime → Change runtime type, then rerun setup and preflight.')

run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('GPU pilot results:', RUN_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
], Path(RUN_ROOT) / 'console.log')

## 4. Inspect the finished GPU-pilot outputs

Run this only after the GPU pilot cell has completed. It displays the saved numerical summary, loss curves, three predicted committors, finite-difference references, and pointwise errors.

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## Resume an interrupted GPU pilot

Use this only if the GPU pilot started and then stopped before finishing. In a fresh runtime, run setup and pass the CPU preflight first. Then paste the Drive folder path printed by the interrupted pilot. The saved checkpoints continue the existing run. The resume cell is locked until the CPU preflight passes.

In [ ]:
# Run this only if the GPU pilot stopped before it finished.
if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('Resume blocked: rerun setup and pass the CPU preflight in this runtime first.')
# Paste the exact Drive folder path printed at the start of that pilot.
from pathlib import Path
import subprocess

RUN_ROOT = input('Paste the interrupted pilot Drive folder path, then press Enter: ').strip()
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError('That folder is missing comparison_config.json. Paste the exact folder path printed by the interrupted pilot.')

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--resume',
], Path(RUN_ROOT) / 'console.log', append=True)
print('The interrupted pilot has resumed and completed.')